# 04 — Análise exploratória
Resultados por ano, UF, pista, horário, tipo e causa; gráficos públicos reproduzíveis em \`figures/\`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
for p in [DATA/"raw", DATA/"processed", OUT/"auditoria", OUT/"eda", OUT/"estatistica", OUT/"modelos", OUT/"interpretabilidade", OUT/"modelo_final"]:
    p.mkdir(parents=True,exist_ok=True)
SILVER = DATA/"processed"/"prf_acidentes_silver_2020_2025.parquet"
import matplotlib.pyplot as plt
df=pd.read_parquet(SILVER)
kpis=pd.DataFrame({"indicador":["acidentes","graves","nao_graves","mortos","feridos_graves","feridos_leves"],"valor":[len(df),df.acidente_grave.sum(),len(df)-df.acidente_grave.sum(),df.mortos.sum(),df.feridos_graves.sum(),df.feridos_leves.sum()]})
kpis.to_csv(OUT/"eda"/"kpis.csv",index=False)
anual=df.groupby("ano").agg(acidentes=("id","size"),graves=("acidente_grave","sum"),mortos=("mortos","sum")).reset_index()
anual["taxa_grave"]=100*anual.graves/anual.acidentes
display(anual)
anual.to_csv(OUT/"eda"/"acidentes_por_ano.csv",index=False)

In [ ]:
def resumo(col, minimo=0, base=None):
    x=df if base is None else base
    t=x.groupby(col,dropna=False).agg(acidentes=("id","size"),graves=("acidente_grave","sum")).reset_index()
    t["taxa_grave"]=100*t.graves/t.acidentes
    return t[t.acidentes>=minimo].sort_values("taxa_grave",ascending=False)
for nome,col in [("tipo_acidente","tipo_acidente"),("uf","uf"),("tipo_pista","tipo_pista"),("fase_dia","fase_dia"),("hora","hora"),("br","br"),("clima","condicao_metereologica")]:
    t=resumo(col)
    t.to_csv(OUT/"eda"/(nome+".csv"),index=False)
    print(nome);display(t.head(10))
causas=resumo("causa_acidente",base=df[df.ano>=2021])
causas.to_csv(OUT/"eda"/"causas_2021_2025.csv",index=False)